# Multi-Image Camera Calibration

This notebook demonstrates multi-image camera calibration using tetra3rs. We solve 10 TESS images from the same CCD across different sectors, iteratively refining a shared camera model (focal length, optical center, polynomial distortion) to achieve sub-10" RMSE.

In [ ]:
import tetra3rs as t3rs
import numpy as np
from astropy.io import fits
import datetime
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from IPython.display import HTML


pio.renderers.default = "notebook"  # or 'jupyterlab'

sectors = [1, 2, 3, 4, 5, 6, 13, 14, 15, 17]
# sectors = [1, 4, 5, 13, 14, 15, 17]
fnames = [f"../../data/tess_same_ccd/sector{sector:02d}_cam1_ccd1.fits" for sector in sectors]


In [ ]:
solver = t3rs.SolverDatabase.generate_from_hipparcos(
    "../../data/hip2.dat",
    max_fov_deg=14.0,
    pattern_max_error=0.005,
    lattice_field_oversampling=100,
    patterns_per_lattice_field=500,
    verification_stars_per_fov=3000,
    epoch_proper_motion_year=2018,
)

In [ ]:
# Get frame times and center pixel pointing from the FITS headers
import warnings
from astropy.utils.exceptions import AstropyWarning

# Get the RA & Dec of center pixel of all the images, from the FITS headers
ra_dec_fits = []
frame_times = []
frame_velocities = []
for fname in fnames:
    with fits.open(fname) as hdul:
        # Get the center of integration
        tstart = datetime.datetime.strptime(
            hdul[0].header["DATE-OBS"], "%Y-%m-%dT%H:%M:%S.%f"
        )
        tstop = datetime.datetime.strptime(
            hdul[0].header["DATE-END"], "%Y-%m-%dT%H:%M:%S.%f"
        )
        tmid = tstart + (tstop - tstart) / 2
        frame_times.append(tmid)
        frame_velocities.append(t3rs.earth_barycentric_velocity(tmid))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", AstropyWarning)
            wcs = WCS(hdul[1].header)
        # Account for 44 pixel offset to science region of the image
        ra_dec = wcs.pixel_to_world(1024 + 44, 1024)
        ra_dec_fits.append((ra_dec.ra.degree, ra_dec.dec.degree))

In [ ]:
# solve centroids for all images
cenarr = []
for idx in range(len(fnames)):
    fname = fnames[idx]
    img: np.ndarray = fits.getdata(fname, ext=1, memmap=False)  # type: ignore
    img = img.astype(np.float32)[:2048, 44:2092]  # 2048x2048 science region
    cenarr.append(
        t3rs.extract_centroids(
            img,
            sigma_threshold=300,
            min_pixels=4,
            max_pixels=10000,
            local_bg_block_size=16,
            max_elongation=6.0,
        )
    )
cenarr = [c.centroids for c in cenarr]

In [ ]:
# Solve on all images with centroids from above
# We iterate solve on multiple images, do a camera calibration,
# use the camera calibration to improve a re-solve, then
# re-do camera calibration with the improved solve.
camera_model = None

pass_configs = [
    (0.01, 10, 3, 0.5),
    (0.003, 10, 5, 0.5),
    (0.002, 10, 6, 0.5),
    (0.001, 10, 6, 0.5),
]

for passidx in range(len(pass_configs)):
    resarr = []

    match_radius, refine_iterations, camera_model_order, fov_error_deg = pass_configs[
        passidx
    ]

    for idx, cen in enumerate(cenarr):
        resarr.append(
            solver.solve_from_centroids(
                cen,
                fov_estimate_deg=camera_model.fov_deg if camera_model else 11.8,
                fov_max_error_deg=fov_error_deg,
                image_shape=(2048, 2048),
                match_radius=match_radius,
                match_threshold=1.0e-5,
                refine_iterations=refine_iterations,
                camera_model=camera_model,
                # observer_velocity_km_s=frame_velocities[idx],
            )
        )

    camera_model = solver.calibrate_camera(
        resarr, cenarr, image_shape=(2048, 2048), order=camera_model_order
    ).camera_model

# Build results table

arcsec_per_px = resarr[0].fov_deg * 3600 / 2048

rows = []
for idx, res in enumerate(resarr):
    ra_dec_model = res.pixel_to_world(0, 0)
    coord_fits = SkyCoord(
        ra_dec_fits[idx][0] * u.degree, ra_dec_fits[idx][1] * u.degree
    )
    coord_model = SkyCoord(ra_dec_model[0] * u.degree, ra_dec_model[1] * u.degree)
    separation = coord_fits.separation(coord_model).arcsecond
    rmse_px = res.rmse_arcsec / arcsec_per_px
    rows.append(
        (
            sectors[idx],
            res.num_matches,
            f"{res.rmse_arcsec:.2f}",
            f"{rmse_px:.3f}",
            f"{separation:.2f}",
            f"{res.solve_time_ms:.1f}",
        )
    )

header = '<tr><th>Sector</th><th>Matches</th><th>RMSE (")</th><th>RMSE (px)</th><th>vs FITS WCS (")</th><th>Solve (ms)</th></tr>'
body = "".join(
    f"<tr><td>{s}</td><td>{n}</td><td>{r}</td><td>{rpx}</td><td>{sep}</td><td>{t}</td></tr>"
    for s, n, r, rpx, sep, t in rows
)
display(HTML(f"<table>{header}{body}</table>"))
# print(f"\nCamera model: {camera_model}")

In [ ]:
from plotly.subplots import make_subplots

ncols, nrows = 3, 4
ds = 4  # downsample factor for display
theta = np.linspace(0, 2 * np.pi, 100)

fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=[f"Sector {s}" for s in sectors],
    vertical_spacing=0.04,
    horizontal_spacing=0.02,
)

for idx, fname in enumerate(fnames):
    row = idx // ncols + 1
    col = idx % ncols + 1

    img: np.ndarray = fits.getdata(fname, ext=1, memmap=False)  # type: ignore
    img = img[:2048, 44:2092].astype(np.float32)
    vmin, vmax = np.percentile(img, 1), np.percentile(img, 99.2)
    img_ds = img[::ds, ::ds]

    img_fig = px.imshow(
        img_ds,
        binary_string=True,
        color_continuous_scale="gray",
        zmin=vmin,
        zmax=vmax,
    )
    fig.add_trace(img_fig.data[0], row=row, col=col)

    # Get matched centroid and star pixel positions
    res = resarr[idx]
    cen_indices = res.matched_centroids
    cat_ids = res.matched_catalog_ids

    # Get catalog star RA/Dec and project to pixel coordinates
    star_ra = np.array([solver.get_star_by_id(int(cid)).ra_deg for cid in cat_ids])
    star_dec = np.array([solver.get_star_by_id(int(cid)).dec_deg for cid in cat_ids])
    star_xy = res.world_to_pixel(
        star_ra, star_dec
    )  # (x_array, y_array) centered coords

    # Draw blue x markers at matched star locations
    sx = (star_xy[0] + 1024) / ds
    sy = (star_xy[1] + 1024) / ds
    fig.add_trace(
        go.Scatter(
            x=sx,
            y=sy,
            mode="markers",
            marker=dict(color="cyan", symbol="cross", size=2, line=dict(width=0.25)),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )

    # Draw centroid ellipses
    for j, ci in enumerate(cen_indices):
        c = cenarr[idx][int(ci)]
        cx = (c.x + 1024) / ds
        cy = (c.y + 1024) / ds
        a = 6 * np.sqrt(c.cov[0, 0]) / ds
        b = 6 * np.sqrt(c.cov[1, 1]) / ds
        angle = (
            0.5 * np.arctan2(2 * c.cov[1, 0], c.cov[0, 0] - c.cov[1, 1]) * 180 / np.pi
            + 90
        )
        x = a * np.cos(theta)
        y = b * np.sin(theta)
        x_rot = x * np.cos(np.radians(angle)) - y * np.sin(np.radians(angle))
        y_rot = x * np.sin(np.radians(angle)) + y * np.cos(np.radians(angle))
        fig.add_trace(
            go.Scatter(
                x=cx + x_rot,
                y=cy + y_rot,
                mode="lines",
                line=dict(color="yellow", width=1),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )


fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_xaxes(range=[0, 2048 / 4], scaleanchor="y", scaleratio=1)
fig.update_yaxes(range=[2048 / 4, 0], scaleanchor="x", scaleratio=1)
fig.update_layout(
    width=900,
    height=900 * 4.6 / 3,
    showlegend=False,
    title_text="TESS Images with Matched Centroids",
    margin=dict(l=10, r=10, b=10),
)
fig.show()